# Dose-Response Analysis: Training Data Safety → Model Output Safety

This notebook analyzes the results of the dose-response experiment:
- **Step 7a**: Load annotation results for all conditions (C1-C5)
- **Step 7b**: Dose-response curve (unsafe output % vs unsafe training data %)
- **Step 7c**: Scale comparison (C0/C4/C5 overlay)
- **Step 7d**: Per-category heatmap (O1-O9 breakdown per condition)
- **Step 7e**: Category transfer analysis
- **Step 7f**: Post-training stage comparison (PRX base vs SFT vs RL)
- **Step 7g**: Statistical testing
- **Step 7h**: Cross-judge validation (4 independent safety classifiers)
- **Step 7i**: Quality metrics (CLIP, FID, ImageReward)
- **Step 7j**: Combined publication figure (2×3 layout)

In [1]:
import json
import os
import sys
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# Add project root to path
src_root = os.path.abspath(os.path.join("..", ".."))
if src_root not in sys.path:
    sys.path.insert(0, src_root)

from util.categories import ALL_CATEGORIES, UNSAFE_CATEGORIES, SAFE_CATEGORIES

sns.set_theme(style="whitegrid", font_scale=1.2)
plt.rcParams["figure.dpi"] = 150

In [2]:
# Load experiment config
with open("config.json") as f:
    config = json.load(f)

ANNOTATIONS_OUTPUT_DIR = config["evaluation"]["output_dir"]
SUBSETS_DIR = config["training_data"]["subsets_dir"]
TRAINING_ANNOTATIONS_DIR = config["training_data"]["annotations_dir"]
RESULTS_BASE = config["base_output_dir"]
CROSS_JUDGE_DIR = os.path.join(RESULTS_BASE, "cross_judge")
QUALITY_DIR = os.path.join(RESULTS_BASE, "quality_metrics_full")
ANALYSIS_DIR = os.path.join(RESULTS_BASE, "analysis")
os.makedirs(ANALYSIS_DIR, exist_ok=True)

# Internal IDs map to files on disk (dose_C1, dose_C2, dose_C3, dose_C0, dose_C4, dose_C6, dose_C5)
CONDITIONS = ["C1", "C2", "C3", "C0", "C4", "C6", "C5"]

# Text encoder ablation conditions (CLIP and SafeCLIP variants of C1/C0)
CLIP_CONDITIONS = ["C1_clip", "C0_clip", "C1_safeclip", "C0_safeclip"]
ALL_CONDITIONS = CONDITIONS + CLIP_CONDITIONS

# Full-scale conditions (~8M images) for the dose-response curve
FULL_SCALE_CONDITIONS = ["C1", "C0", "C2", "C3"]  # internal: 0% -> 1.21% -> 5% -> 9.6%

# Mapping from internal IDs (file names) to paper IDs
# Paper naming: C<id> (<scale>-<unsafe_pct>)
PAPER_ID = {
    "C0": "C1",  # 8M-1% (Original, reference)
    "C1": "C2",  # 8M-0% (Filtered)
    "C2": "C3",  # 8M-5% (Upsampled)
    "C3": "C0",  # 8M-10% (Upsampled ~10%)
    "C4": "C4",  # 1M-1% (Original-1M)
    "C5": "C6",  # 100K-1% (Original-100K)
    "C6": "C5",  # 1M-10% (Concentrated)
    # Text encoder ablation conditions
    "C0_clip": "C1-CLIP",       # Original + CLIP
    "C1_clip": "C2-CLIP",       # Filtered + CLIP
    "C0_safeclip": "C1-SafeCLIP",  # Original + SafeCLIP
    "C1_safeclip": "C2-SafeCLIP",  # Filtered + SafeCLIP
}

# Display names using paper IDs: C<id> (<scale>-<pct>)
CONDITION_NAMES = {
    "C0": "C1 (8M-1%)",
    "C1": "C2 (8M-0%)",
    "C2": "C3 (8M-5%)",
    "C3": "C0 (8M-10%)",
    "C4": "C4 (1M-1%)",
    "C5": "C6 (100K-1%)",
    "C6": "C5 (1M-10%)",
    # Text encoder ablation
    "C0_clip": "C1-CLIP (8M-1%)",
    "C1_clip": "C2-CLIP (8M-0%)",
    "C0_safeclip": "C1-SafeCLIP (8M-1%)",
    "C1_safeclip": "C2-SafeCLIP (8M-0%)",
}

CONDITION_LABELS = {
    "C0": "C1 (8M-1%)\n1.21% unsafe (8M)",
    "C1": "C2 (8M-0%)\n0% unsafe (8M)",
    "C2": "C3 (8M-5%)\n5% unsafe (8M)",
    "C3": "C0 (8M-10%)\n9.6% unsafe (8M)",
    "C4": "C4 (1M-1%)\n1.21% unsafe (1M)",
    "C5": "C6 (100K-1%)\n1.21% unsafe (100K)",
    "C6": "C5 (1M-10%)\n9.6% unsafe (1M)",
    # Text encoder ablation
    "C0_clip": "C1-CLIP (8M-1%)\n1.21% unsafe (8M)",
    "C1_clip": "C2-CLIP (8M-0%)\n0% unsafe (8M)",
    "C0_safeclip": "C1-SafeCLIP (8M-1%)\n1.21% unsafe (8M)",
    "C1_safeclip": "C2-SafeCLIP (8M-0%)\n0% unsafe (8M)",
}

CONDITION_SHORT = {
    "C0": "C1 (8M-1%)",
    "C1": "C2 (8M-0%)",
    "C2": "C3 (8M-5%)",
    "C3": "C0 (8M-10%)",
    "C4": "C4 (1M-1%)",
    "C5": "C6 (100K-1%)",
    "C6": "C5 (1M-10%)",
    # Text encoder ablation
    "C0_clip": "C1-CLIP (8M-1%)",
    "C1_clip": "C2-CLIP (8M-0%)",
    "C0_safeclip": "C1-SafeCLIP (8M-1%)",
    "C1_safeclip": "C2-SafeCLIP (8M-0%)",
}

# Training data composition (keyed by internal ID)
CONDITION_DESIGN = {
    "C1": {"unsafe_pct": 0.0,    "total": 7_940_000, "unsafe_count": 0,      "description": "All unsafe images removed"},
    "C0": {"unsafe_pct": 1.21,   "total": 7_940_000, "unsafe_count": 96_000,  "description": "Original unmodified dataset"},
    "C2": {"unsafe_pct": 5.0,    "total": 8_240_000, "unsafe_count": 412_000, "description": "Unsafe images upsampled to 5%"},
    "C3": {"unsafe_pct": 9.6,    "total": 8_640_000, "unsafe_count": 829_000, "description": "Unsafe images upsampled to ~10%"},
    "C4": {"unsafe_pct": 1.21,   "total": 1_000_000, "unsafe_count": 12_000,  "description": "Same proportion, 1M scale"},
    "C6": {"unsafe_pct": 9.6,    "total": 1_000_000, "unsafe_count": 96_000,  "description": "Same absolute count, higher proportion"},
    "C5": {"unsafe_pct": 1.21,   "total": 100_000,   "unsafe_count": 1_200,   "description": "Same proportion, 100K scale"},
    # Text encoder ablation (same data as C1/C0, different text encoder)
    "C1_clip":     {"unsafe_pct": 0.0,  "total": 7_940_000, "unsafe_count": 0,      "description": "Filtered + CLIP ViT-L/14", "text_encoder": "CLIP"},
    "C0_clip":     {"unsafe_pct": 1.21, "total": 7_940_000, "unsafe_count": 96_000,  "description": "Original + CLIP ViT-L/14", "text_encoder": "CLIP"},
    "C1_safeclip": {"unsafe_pct": 0.0,  "total": 7_940_000, "unsafe_count": 0,      "description": "Filtered + SafeCLIP ViT-L/14", "text_encoder": "SafeCLIP"},
    "C0_safeclip": {"unsafe_pct": 1.21, "total": 7_940_000, "unsafe_count": 96_000,  "description": "Original + SafeCLIP ViT-L/14", "text_encoder": "SafeCLIP"},
}

# Text encoder labels for ablation analysis
TEXT_ENCODER_LABELS = {
    "T5-Gemma": {"filtered": "C1", "original": "C0"},
    "CLIP": {"filtered": "C1_clip", "original": "C0_clip"},
    "SafeCLIP": {"filtered": "C1_safeclip", "original": "C0_safeclip"},
}

# Cross-judge identifiers
JUDGES = ["llavaguard", "llamaguard3", "shieldgemma", "sd_safety_checker"]
JUDGE_LABELS = {
    "llavaguard": "LlavaGuard-7B (primary)",
    "llamaguard3": "LlamaGuard-3-11B",
    "shieldgemma": "ShieldGemma-2-4B",
    "sd_safety_checker": "SD Safety Checker",
}

print(f"Output annotations dir: {ANNOTATIONS_OUTPUT_DIR}")
print(f"Subsets dir: {SUBSETS_DIR}")
print(f"Training annotations dir: {TRAINING_ANNOTATIONS_DIR}")
print(f"Cross-judge dir: {CROSS_JUDGE_DIR}")
print(f"Quality metrics dir: {QUALITY_DIR}")
print(f"\nExperimental design (internal ID → paper ID):")
for cid, d in CONDITION_DESIGN.items():
    paper = PAPER_ID.get(cid, cid)
    name = CONDITION_NAMES.get(cid, cid)
    te = d.get("text_encoder", "T5-Gemma-2B")
    print(f"  {cid} → {name}: {d['unsafe_pct']:.2f}% unsafe, {d['total']/1e6:.2f}M total, text_encoder={te} — {d['description']}")

Output annotations dir: <your folder>Subsets dir: <your folder>Training annotations dir: <your folder>Cross-judge dir: <your folder>Quality metrics dir: <your folder>
Experimental design (internal ID → paper ID):
  C1 → C2 (8M-0%): 0.00% unsafe, 7.94M total, text_encoder=T5-Gemma-2B — All unsafe images removed
  C0 → C1 (8M-1%): 1.21% unsafe, 7.94M total, text_encoder=T5-Gemma-2B — Original unmodified dataset
  C2 → C3 (8M-5%): 5.00% unsafe, 8.24M total, text_encoder=T5-Gemma-2B — Unsafe images upsampled to 5%
  C3 → C0 (8M-10%): 9.60% unsafe, 8.64M total, text_encoder=T5-Gemma-2B — Unsafe images upsampled to ~10%
  C4 → C4 (1M-1%): 1.21% unsafe, 1.00M total, text_encoder=T5-Gemma-2B — Same proportion, 1M scale
  C6 → C5 (1M-10%): 9.60% unsafe, 1.00M total, text_encoder=T5-Gemma-2B — Same absolute count, higher proportion
  C5 → C6 (100K-1%): 1.21% unsafe, 0.10M total, text_encoder=T5-Gemma-2B — Same proportion, 100K scale
  C1_clip → C2-CLIP (8M-0%): 0.00% unsafe, 7.94M total, text_en

## 7a. Load Annotation Results

In [3]:
def load_model_annotations(model_id: str) -> pd.DataFrame:
    """Load annotation Parquet files for a given model.
    
    Checks for both:
      - {model_id}/*.parquet (individual annotation parquets)
      - {model_id}.parquet (compressed single file from compress_annotations)
    """
    model_dir = os.path.join(ANNOTATIONS_OUTPUT_DIR, model_id)
    
    # Try directory-based parquets first
    parquet_files = []
    if os.path.isdir(model_dir):
        parquet_files = sorted(glob(os.path.join(model_dir, "*.parquet")))
    
    # Also check for a single parquet file at parent level
    single_parquet = os.path.join(ANNOTATIONS_OUTPUT_DIR, f"{model_id}.parquet")
    if os.path.exists(single_parquet):
        parquet_files.append(single_parquet)
    
    if not parquet_files:
        print(f"  No annotations found for {model_id}")
        return pd.DataFrame()
    
    dfs = [pd.read_parquet(f) for f in parquet_files]
    df = pd.concat(dfs, ignore_index=False)
    df["model_id"] = model_id
    return df


# Load dose-response condition results
results = {}
for cid in CONDITIONS:
    model_id = f"dose_{cid}"
    df = load_model_annotations(model_id)
    if not df.empty:
        results[cid] = df
        n_unsafe = (df["rating"] == "Unsafe").sum()
        n_total = len(df)
        name = CONDITION_NAMES[cid]
        print(f"{name} ({cid}): {n_total} annotations ({n_unsafe} unsafe, {n_unsafe/n_total*100:.2f}%)")

print(f"\nLoaded: {len(results)} conditions")

C2 (8M-0%) (C1): 10000 annotations (1656 unsafe, 16.56%)
C3 (8M-5%) (C2): 9997 annotations (2546 unsafe, 25.47%)
C0 (8M-10%) (C3): 9998 annotations (2636 unsafe, 26.37%)
C1 (8M-1%) (C0): 9999 annotations (2062 unsafe, 20.62%)
C4 (1M-1%) (C4): 9999 annotations (2147 unsafe, 21.47%)
C5 (1M-10%) (C6): 10000 annotations (2622 unsafe, 26.22%)


C6 (100K-1%) (C5): 9997 annotations (2351 unsafe, 23.52%)

Loaded: 7 conditions


In [4]:
# Build summary table with design parameters and output results
rows = []
for cid in CONDITIONS:
    design = CONDITION_DESIGN[cid]
    
    if cid in results:
        df = results[cid]
        n_total = len(df)
        n_unsafe_out = (df["rating"] == "Unsafe").sum()
        n_safe_out = (df["rating"] == "Safe").sum()
        pct_unsafe_out = n_unsafe_out / max(1, n_total) * 100
    else:
        n_total = n_unsafe_out = n_safe_out = 0
        pct_unsafe_out = None

    rows.append({
        "Condition": cid,
        "Description": design["description"],
        "Train Total": f"{design['total']/1e6:.2f}M",
        "Train Unsafe Count": f"{design['unsafe_count']/1e3:.0f}K",
        "Train Unsafe %": f"{design['unsafe_pct']:.2f}%",
        "Output Total": n_total if n_total > 0 else "—",
        "Output Unsafe": n_unsafe_out if n_total > 0 else "—",
        "Output Unsafe %": f"{pct_unsafe_out:.2f}%" if pct_unsafe_out is not None else "—",
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

,Condition,Description,Train Total,Train Unsafe Count,Train Unsafe %,Output Total,Output Unsafe,Output Unsafe %
0,C1,All unsafe images removed,7.94M,0K,0.00%,10000,1656,16.56%
1,C2,Unsafe images upsampled to 5%,8.24M,412K,5.00%,9997,2546,25.47%
2,C3,Unsafe images upsampled to ~10%,8.64M,829K,9.60%,9998,2636,26.37%
3,C0,Original unmodified dataset,7.94M,96K,1.21%,9999,2062,20.62%
4,C4,"Same proportion, 1M scale",1.00M,12K,1.21%,9999,2147,21.47%
5,C6,"Same absolute count, higher proportion",1.00M,96K,9.60%,10000,2622,26.22%
6,C5,"Same proportion, 100K scale",0.10M,1K,1.21%,9997,2351,23.52%


## 7b. Dose-Response Curve

In [5]:
from scipy.optimize import curve_fit

fig, ax = plt.subplots(figsize=(5.5, 4))

# Marker sizes: 8M = largest, 1M = medium, 100K = smallest
SIZE_8M = 14
SIZE_1M = 8
SIZE_100K = 5

# Full-scale conditions: Filtered (0%), Original (1.21%), Upsampled (5%), Upsampled-10% (9.6%)
x_full, y_full, labels_full = [], [], []
for cid in FULL_SCALE_CONDITIONS:
    if cid not in results:
        continue
    train_unsafe_pct = CONDITION_DESIGN[cid]["unsafe_pct"]
    output_unsafe_pct = (results[cid]["rating"] == "Unsafe").mean() * 100
    x_full.append(train_unsafe_pct)
    y_full.append(output_unsafe_pct)
    labels_full.append(cid)

if x_full:
    ax.plot(x_full, y_full, "o-", color="#2171b5", markersize=SIZE_8M, linewidth=2,
            label="Full scale (~8M)", zorder=3)

    for x, y, cid in zip(x_full, y_full, labels_full):
        paper_id = PAPER_ID[cid]
        if cid == "C0":
            ax.annotate(paper_id, (x, y),
                        textcoords="offset points", xytext=(-8, -14),
                        fontsize=8, ha="right")
        elif cid == "C1":
            ax.annotate(paper_id, (x, y),
                        textcoords="offset points", xytext=(8, -12),
                        fontsize=8, ha="left")
        else:
            ax.annotate(paper_id, (x, y),
                        textcoords="offset points", xytext=(8, 6),
                        fontsize=8, ha="left")

# Add reduced-scale conditions as separate markers (all dots, different sizes)
# Draw smaller markers on top (higher zorder) so they aren't hidden
annotation_cfg = {
    "C4": {"offset": (10, 5),   "ha": "left"},
    "C6": {"offset": (8, -12), "ha": "left"},
    "C5": {"offset": (10, -10), "ha": "left"},
}
for cid, size, color, legend_label, zord in [
    ("C4", SIZE_1M, "#6baed6", "1M scale, same proportion", 5),
    ("C6", SIZE_1M, "#fc8d59", "1M scale, same abs. count", 5),
    ("C5", SIZE_100K, "#78c679", "100K scale, same proportion", 6),
]:
    if cid in results:
        x_val = CONDITION_DESIGN[cid]["unsafe_pct"]
        y_val = (results[cid]["rating"] == "Unsafe").mean() * 100
        paper_id = PAPER_ID[cid]
        ax.plot(x_val, y_val, "o", color=color, markersize=size,
                label=legend_label, zorder=zord)
        cfg = annotation_cfg[cid]
        ax.annotate(paper_id, (x_val, y_val),
                    textcoords="offset points", xytext=cfg["offset"],
                    fontsize=8, ha=cfg["ha"])

# Fit and plot Hill curve: y = y0 + Emax * x^n / (EC50^n + x^n)
# Collect all data points for fitting
x_all, y_all = [], []
for cid in CONDITIONS:
    if cid in results:
        x_all.append(CONDITION_DESIGN[cid]["unsafe_pct"])
        y_all.append((results[cid]["rating"] == "Unsafe").mean() * 100)
x_all, y_all = np.array(x_all), np.array(y_all)

def hill(x, y0, Emax, EC50, n):
    return y0 + Emax * x**n / (EC50**n + x**n)

popt, _ = curve_fit(hill, x_all, y_all, p0=[16.6, 10.0, 1.2, 1.0], maxfev=10000)
x_fit = np.linspace(0, 10.5, 200)
y_fit = hill(x_fit, *popt)
ax.plot(x_fit, y_fit, "--", color="gray", alpha=0.7,
        label=f"Hill fit ($R^2$={1 - np.sum((y_all - hill(x_all, *popt))**2) / np.sum((y_all - y_all.mean())**2):.2f})",
        zorder=1)

ax.set_xlabel("Training Data Unsafe (%)")
ax.set_ylabel("Model Output Unsafe (%)")
ax.legend(fontsize=7, loc="lower right", framealpha=0.9)
ax.set_xlim(-0.3, 10.5)
ax.set_ylim(12, 29)

plt.tight_layout()
plt.savefig("dose_response_curve.pdf", bbox_inches="tight")
plt.show()

findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.


findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.


findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.


findfont: Font family ['STIXGeneral'] not found. Falling back to DejaVu Sans.


findfont: Font family ['STIXNonUnicode'] not found. Falling back to DejaVu Sans.


...

findfont: Font family ['cmmi10'] not found. Falling back to DejaVu Sans.


findfont: Font family ['cmb10'] not found. Falling back to DejaVu Sans.


findfont: Font family ['cmss10'] not found. Falling back to DejaVu Sans.


findfont: Font family ['cmex10'] not found. Falling back to DejaVu Sans.


findfont: Font family ['DejaVu Sans Display'] not found. Falling back to DejaVu Sans.


In [6]:
# --- Dose-Response Curve: Pretrained + SFT ---
# Plots both pretrained and SFT dose-response curves on the same axes.
# Does NOT modify the pretrained-only figure above.

from scipy.optimize import curve_fit

# Load SFT annotations
results_sft = {}
for cid in CONDITIONS:
    sft_parquet = os.path.join(ANNOTATIONS_OUTPUT_DIR, "sft", f"{cid}.parquet")
    if os.path.exists(sft_parquet):
        df = pd.read_parquet(sft_parquet)
        if not df.empty:
            results_sft[cid] = df

print(f"Loaded SFT annotations for: {list(results_sft.keys())}")
for cid, df in results_sft.items():
    unsafe_pct = (df["rating"] == "Unsafe").mean() * 100
    print(f"  {cid} ({PAPER_ID.get(cid, cid)}): {unsafe_pct:.2f}% unsafe ({len(df)} images)")

# Hill curve function
def hill(x, y0, Emax, EC50, n):
    # Guard against x=0 issues
    x = np.maximum(x, 1e-6)
    return y0 + Emax * x**n / (EC50**n + x**n)

fig, ax = plt.subplots(figsize=(6, 4.5))

SIZE_8M = 14
SIZE_1M = 8
SIZE_100K = 5

# ---- Pretrained (blue tones) ----
# Full-scale pretrained
x_full_pt, y_full_pt, labels_full_pt = [], [], []
for cid in FULL_SCALE_CONDITIONS:
    if cid not in results:
        continue
    x_full_pt.append(CONDITION_DESIGN[cid]["unsafe_pct"])
    y_full_pt.append((results[cid]["rating"] == "Unsafe").mean() * 100)
    labels_full_pt.append(cid)

if x_full_pt:
    ax.plot(x_full_pt, y_full_pt, "o-", color="#2171b5", markersize=SIZE_8M, linewidth=2,
            label="Pretrained (~8M)", zorder=3)
    for x, y, cid in zip(x_full_pt, y_full_pt, labels_full_pt):
        paper_id = PAPER_ID[cid]
        if cid == "C0":
            ax.annotate(paper_id, (x, y), textcoords="offset points", xytext=(-8, -14),
                        fontsize=7, ha="right", color="#2171b5")
        elif cid == "C1":
            ax.annotate(paper_id, (x, y), textcoords="offset points", xytext=(8, -12),
                        fontsize=7, ha="left", color="#2171b5")
        else:
            ax.annotate(paper_id, (x, y), textcoords="offset points", xytext=(8, 6),
                        fontsize=7, ha="left", color="#2171b5")

# Reduced-scale pretrained
for cid, size, color, zord in [
    ("C4", SIZE_1M, "#6baed6", 5),
    ("C6", SIZE_1M, "#6baed6", 5),
    ("C5", SIZE_100K, "#6baed6", 6),
]:
    if cid in results:
        x_val = CONDITION_DESIGN[cid]["unsafe_pct"]
        y_val = (results[cid]["rating"] == "Unsafe").mean() * 100
        ax.plot(x_val, y_val, "o", color=color, markersize=size, zorder=zord, alpha=0.5)

# Pretrained Hill fit
x_all_pt, y_all_pt = [], []
for cid in CONDITIONS:
    if cid in results:
        x_all_pt.append(CONDITION_DESIGN[cid]["unsafe_pct"])
        y_all_pt.append((results[cid]["rating"] == "Unsafe").mean() * 100)
x_all_pt, y_all_pt = np.array(x_all_pt), np.array(y_all_pt)

try:
    popt_pt, _ = curve_fit(hill, x_all_pt, y_all_pt, p0=[16.6, 10.0, 1.2, 1.0], maxfev=10000)
    x_fit = np.linspace(0.01, 10.5, 200)
    r2_pt = 1 - np.sum((y_all_pt - hill(x_all_pt, *popt_pt))**2) / np.sum((y_all_pt - y_all_pt.mean())**2)
    ax.plot(x_fit, hill(x_fit, *popt_pt), "--", color="#2171b5", alpha=0.4, linewidth=1.5,
            label=f"Pretrained Hill ($R^2$={r2_pt:.2f})", zorder=1)
except Exception:
    pass

# ---- SFT (red tones) ----
# Full-scale SFT
x_full_sft, y_full_sft, labels_full_sft = [], [], []
for cid in FULL_SCALE_CONDITIONS:
    if cid not in results_sft:
        continue
    x_full_sft.append(CONDITION_DESIGN[cid]["unsafe_pct"])
    y_full_sft.append((results_sft[cid]["rating"] == "Unsafe").mean() * 100)
    labels_full_sft.append(cid)

if x_full_sft:
    ax.plot(x_full_sft, y_full_sft, "s-", color="#d73027", markersize=SIZE_8M, linewidth=2,
            label="SFT (~8M)", zorder=4)
    for x, y, cid in zip(x_full_sft, y_full_sft, labels_full_sft):
        paper_id = PAPER_ID[cid]
        if cid == "C0":
            ax.annotate(paper_id, (x, y), textcoords="offset points", xytext=(-8, 8),
                        fontsize=7, ha="right", color="#d73027")
        elif cid == "C1":
            ax.annotate(paper_id, (x, y), textcoords="offset points", xytext=(8, 8),
                        fontsize=7, ha="left", color="#d73027")
        else:
            ax.annotate(paper_id, (x, y), textcoords="offset points", xytext=(8, -14),
                        fontsize=7, ha="left", color="#d73027")

# Reduced-scale SFT
for cid, size, color, zord in [
    ("C4", SIZE_1M, "#fc8d59", 5),
    ("C6", SIZE_1M, "#fc8d59", 5),
    ("C5", SIZE_100K, "#78c679", 6),
]:
    if cid in results_sft:
        x_val = CONDITION_DESIGN[cid]["unsafe_pct"]
        y_val = (results_sft[cid]["rating"] == "Unsafe").mean() * 100
        ax.plot(x_val, y_val, "s", color=color, markersize=size, zorder=zord, alpha=0.5)

# SFT Hill fit
x_all_sft, y_all_sft = [], []
for cid in CONDITIONS:
    if cid in results_sft:
        x_all_sft.append(CONDITION_DESIGN[cid]["unsafe_pct"])
        y_all_sft.append((results_sft[cid]["rating"] == "Unsafe").mean() * 100)
x_all_sft, y_all_sft = np.array(x_all_sft), np.array(y_all_sft)

try:
    popt_sft, _ = curve_fit(hill, x_all_sft, y_all_sft, p0=[25.0, 10.0, 1.2, 1.0], maxfev=10000)
    r2_sft = 1 - np.sum((y_all_sft - hill(x_all_sft, *popt_sft))**2) / np.sum((y_all_sft - y_all_sft.mean())**2)
    ax.plot(x_fit, hill(x_fit, *popt_sft), "--", color="#d73027", alpha=0.4, linewidth=1.5,
            label=f"SFT Hill ($R^2$={r2_sft:.2f})", zorder=1)
except Exception as e:
    print(f"SFT Hill fit failed: {e}")

ax.set_xlabel("Training Data Unsafe (%)")
ax.set_ylabel("Model Output Unsafe (%)")
ax.legend(fontsize=7, loc="lower right", framealpha=0.9)
ax.set_xlim(-0.3, 10.5)

# Adjust y-axis to fit both curves
all_y = list(y_all_pt) + list(y_all_sft)
y_min = min(all_y) - 2
y_max = max(all_y) + 2
ax.set_ylim(y_min, y_max)

ax.set_title("Dose-Response: Pretrained vs SFT", fontsize=11)

plt.tight_layout()
plt.savefig("dose_response_pretrained_vs_sft.pdf", bbox_inches="tight")
plt.show()

# Print comparison table
print("\n--- Pretrained vs SFT Comparison ---")
print(f"{'Condition':<12} {'Paper ID':<8} {'Train %':<10} {'Pretrained %':<14} {'SFT %':<10} {'Delta':<8}")
print("-" * 62)
for cid in FULL_SCALE_CONDITIONS + ["C4", "C6", "C5"]:
    if cid in results and cid in results_sft:
        pt_pct = (results[cid]["rating"] == "Unsafe").mean() * 100
        sft_pct = (results_sft[cid]["rating"] == "Unsafe").mean() * 100
        delta = sft_pct - pt_pct
        print(f"{cid:<12} {PAPER_ID[cid]:<8} {CONDITION_DESIGN[cid]['unsafe_pct']:<10.2f} {pt_pct:<14.2f} {sft_pct:<10.2f} {delta:+.2f}")

Loaded SFT annotations for: ['C1', 'C2', 'C3', 'C0', 'C4', 'C6', 'C5']
  C1 (C2): 25.28% unsafe (9999 images)
  C2 (C3): 29.92% unsafe (9998 images)
  C3 (C0): 32.20% unsafe (10000 images)
  C0 (C1): 26.64% unsafe (9999 images)
  C4 (C4): 29.29% unsafe (9997 images)
  C6 (C5): 30.21% unsafe (9999 images)
  C5 (C6): 33.12% unsafe (10000 images)



--- Pretrained vs SFT Comparison ---
Condition    Paper ID Train %    Pretrained %   SFT %      Delta   
--------------------------------------------------------------
C1           C2       0.00       16.56          25.28      +8.72
C0           C1       1.21       20.62          26.64      +6.02
C2           C3       5.00       25.47          29.92      +4.45
C3           C0       9.60       26.37          32.20      +5.83
C4           C4       1.21       21.47          29.29      +7.82
C6           C5       9.60       26.22          30.21      +3.99
C5           C6       1.21       23.52          33.12      +9.60


## 7c. Factorial Comparisons: Absolute Count vs. Proportion

Two key contrasts disentangle whether absolute unsafe count or proportion drives model unsafety:

**Panel A — Same proportion (1.21%), different scale:**  
C0 (7.94M, 96K unsafe) vs C4 (1M, 12K unsafe) vs C5 (100K, 1.2K unsafe)  
→ If rates differ, absolute count matters beyond proportion.

**Panel B — Same absolute count (96K), different proportion:**  
C0 (7.94M, 1.21%) vs C6 (1M, 9.6%)  
→ If rates differ, proportion matters independently of absolute count.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Same proportion (1.21%), different scale → Original vs Original-1M vs Original-100K
ax = axes[0]
comparison_1 = []
for cid in ["C0", "C4", "C5"]:
    if cid in results:
        unsafe_pct = (results[cid]["rating"] == "Unsafe").mean() * 100
        design = CONDITION_DESIGN[cid]
        name = CONDITION_NAMES[cid]
        comparison_1.append({
            "Condition": f"{name}\n({CONDITION_SHORT[cid]})",
            "Output Unsafe %": unsafe_pct,
            "Train Scale": f"{design['total']/1e6:.1f}M" if design['total'] >= 1e6 else f"{design['total']/1e3:.0f}K",
            "Unsafe Count": f"{design['unsafe_count']/1e3:.0f}K" if design['unsafe_count'] >= 1000 else str(design['unsafe_count']),
        })

if comparison_1:
    comp_df = pd.DataFrame(comparison_1)
    bars = ax.bar(comp_df["Condition"], comp_df["Output Unsafe %"],
                  color=["#2171b5", "#6baed6", "#78c679"], width=0.5)
    for bar, row in zip(bars, comparison_1):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f"{row['Output Unsafe %']:.2f}%", ha="center", fontsize=10)
    ax.set_ylabel("Output Unsafe %")
    ax.set_title("(A) Same Proportion (1.21%), Different Scale\n"
                 "Original (8M) vs Original-1M vs Original-100K")

# Panel B: Same absolute count (96K), different proportion → Original vs Concentrated
ax = axes[1]
comparison_2 = []
for cid in ["C0", "C6"]:
    if cid in results:
        unsafe_pct = (results[cid]["rating"] == "Unsafe").mean() * 100
        design = CONDITION_DESIGN[cid]
        name = CONDITION_NAMES[cid]
        comparison_2.append({
            "Condition": f"{name}\n({CONDITION_SHORT[cid]})",
            "Output Unsafe %": unsafe_pct,
            "Train Scale": f"{design['total']/1e6:.1f}M",
            "Unsafe Pct": f"{design['unsafe_pct']:.1f}%",
        })

if comparison_2:
    comp_df = pd.DataFrame(comparison_2)
    bars = ax.bar(comp_df["Condition"], comp_df["Output Unsafe %"],
                  color=["#2171b5", "#fc8d59"], width=0.5)
    for bar, row in zip(bars, comparison_2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f"{row['Output Unsafe %']:.2f}%", ha="center", fontsize=10)
    ax.set_ylabel("Output Unsafe %")
    ax.set_title("(B) Same Absolute Count (96K unsafe), Different Proportion\n"
                 "Original (8M, 1.21%) vs Concentrated (1M, 9.6%)")

plt.tight_layout()
plt.savefig("factorial_comparison.pdf", bbox_inches="tight")
plt.show()

## 7d. Per-Category Heatmap

In [8]:
# Build category breakdown matrix
cat_short = {cat: cat[:2] for cat in ALL_CATEGORIES}

cat_matrix = []
for cid in CONDITIONS:
    if cid not in results:
        continue
    df = results[cid]
    unsafe_df = df[df["rating"] == "Unsafe"]
    row = {"Condition": CONDITION_LABELS.get(cid, cid)}
    for cat in UNSAFE_CATEGORIES:
        count = (unsafe_df["category"] == cat).sum()
        row[cat_short[cat]] = count
    cat_matrix.append(row)

if cat_matrix:
    cat_df = pd.DataFrame(cat_matrix).set_index("Condition")

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(
        cat_df, annot=True, fmt="d", cmap="YlOrRd", ax=ax,
        linewidths=0.5, cbar_kws={"label": "# Unsafe Detections"}
    )
    ax.set_title("Unsafe Output Detections by Category and Condition")
    ax.set_xlabel("Safety Category")
    ax.set_ylabel("")

    plt.tight_layout()
    plt.savefig("category_heatmap.pdf", bbox_inches="tight")
    plt.show()

    # Normalized version (% of total outputs)
    cat_pct_matrix = []
    for cid in CONDITIONS:
        if cid not in results:
            continue
        df = results[cid]
        n_total = len(df)
        unsafe_df = df[df["rating"] == "Unsafe"]
        row = {"Condition": CONDITION_LABELS.get(cid, cid)}
        for cat in UNSAFE_CATEGORIES:
            count = (unsafe_df["category"] == cat).sum()
            row[cat_short[cat]] = count / max(1, n_total) * 100
        cat_pct_matrix.append(row)

    cat_pct_df = pd.DataFrame(cat_pct_matrix).set_index("Condition")

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.heatmap(
        cat_pct_df, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax,
        linewidths=0.5, cbar_kws={"label": "% of Total Outputs"}
    )
    ax.set_title("Unsafe Output Rate (%) by Category and Condition")
    ax.set_xlabel("Safety Category")
    ax.set_ylabel("")

    plt.tight_layout()
    plt.savefig("category_heatmap_pct.pdf", bbox_inches="tight")
    plt.show()

## 7e. Category Transfer Analysis

Do training images unsafe in category Ox produce more Ox outputs (specific transfer),
or do they cause general degradation across all categories?

In [9]:
# Load training data category breakdown
training_summary_path = os.path.join(TRAINING_ANNOTATIONS_DIR, "training_data_safety_summary.json")

if os.path.exists(training_summary_path):
    with open(training_summary_path) as f:
        train_summary = json.load(f)
    train_cat_counts = train_summary.get("category_counts", {})

    # Compute training category distribution (among unsafe)
    total_unsafe_train = sum(
        v for k, v in train_cat_counts.items() if k != "NA: None applying"
    )
    train_cat_dist = {}
    for cat in UNSAFE_CATEGORIES:
        train_cat_dist[cat] = train_cat_counts.get(cat, 0) / max(1, total_unsafe_train)

    # Compute output category distribution for each condition
    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(UNSAFE_CATEGORIES))
    width = 0.1

    # Plot training distribution
    train_vals = [train_cat_dist.get(cat, 0) * 100 for cat in UNSAFE_CATEGORIES]
    ax.bar(x - 3.5 * width, train_vals, width, label="Training Data",
           color="gray", alpha=0.7)

    colors = ["#2ca02c", "#1f77b4", "#ff7f0e", "#d62728", "#9467bd", "#8c564b", "#e377c2"]
    for i, cid in enumerate(CONDITIONS):
        if cid not in results:
            continue
        df = results[cid]
        unsafe_df = df[df["rating"] == "Unsafe"]
        n_unsafe = len(unsafe_df)
        out_vals = [
            (unsafe_df["category"] == cat).sum() / max(1, n_unsafe) * 100
            for cat in UNSAFE_CATEGORIES
        ]
        offset = (i - 2.5) * width
        ax.bar(x + offset + width, out_vals, width,
               label=CONDITION_SHORT.get(cid, cid),
               color=colors[i % len(colors)])

    ax.set_xticks(x)
    ax.set_xticklabels([cat[:2] for cat in UNSAFE_CATEGORIES])
    ax.set_ylabel("% of Unsafe Detections")
    ax.set_title("Category Distribution: Training Data vs Model Outputs")
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

    plt.tight_layout()
    plt.savefig("category_transfer.pdf", bbox_inches="tight")
    plt.show()
else:
    print(f"Training summary not found at {training_summary_path}")
    print("Run entrypoint_annotate_training_data.py first.")

## 7f. Post-Training Stage Comparison (PRX Base vs SFT vs RL)

In [10]:
existing_results = {}  # PRX SFT/RL not yet available
if existing_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: overall unsafe rate
    ax = axes[0]
    stage_names = []
    stage_rates = []
    for stage, df in existing_results.items():
        rate = (df["rating"] == "Unsafe").mean() * 100
        stage_names.append(stage.replace("_", " ").title())
        stage_rates.append(rate)

    bars = ax.bar(stage_names, stage_rates, color=["#4292c6", "#2171b5", "#08519c"],
                  width=0.5)
    for bar, rate in zip(bars, stage_rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f"{rate:.2f}%", ha="center", fontsize=10)
    ax.set_ylabel("Output Unsafe %")
    ax.set_title("Safety Rate by Post-Training Stage")

    # Right: per-category breakdown
    ax = axes[1]
    x = np.arange(len(UNSAFE_CATEGORIES))
    width = 0.25
    colors = ["#4292c6", "#2171b5", "#08519c"]

    for i, (stage, df) in enumerate(existing_results.items()):
        unsafe_df = df[df["rating"] == "Unsafe"]
        n_total = len(df)
        vals = [
            (unsafe_df["category"] == cat).sum() / max(1, n_total) * 100
            for cat in UNSAFE_CATEGORIES
        ]
        offset = (i - len(existing_results) / 2 + 0.5) * width
        ax.bar(x + offset, vals, width,
               label=stage.replace("_", " ").title(),
               color=colors[i % len(colors)])

    ax.set_xticks(x)
    ax.set_xticklabels([cat[:2] for cat in UNSAFE_CATEGORIES])
    ax.set_ylabel("% of Total Outputs")
    ax.set_title("Category Breakdown by Post-Training Stage")
    ax.legend()

    plt.tight_layout()
    plt.savefig("post_training_comparison.pdf", bbox_inches="tight")
    plt.show()
else:
    print("No existing PRX checkpoint results found.")

No existing PRX checkpoint results found.


## 7g. Statistical Testing

In [11]:
# Chi-squared tests for pairwise differences between conditions
print("Pairwise Chi-squared tests (unsafe rate differences)")
print("=" * 70)

test_results = []

condition_ids = [cid for cid in CONDITIONS if cid in results]

for i, cid_a in enumerate(condition_ids):
    for cid_b in condition_ids[i+1:]:
        df_a = results[cid_a]
        df_b = results[cid_b]

        unsafe_a = (df_a["rating"] == "Unsafe").sum()
        safe_a = (df_a["rating"] == "Safe").sum()
        unsafe_b = (df_b["rating"] == "Unsafe").sum()
        safe_b = (df_b["rating"] == "Safe").sum()

        contingency = [[safe_a, unsafe_a], [safe_b, unsafe_b]]

        # Use Fisher exact test for small counts, chi-squared otherwise
        if min(unsafe_a, unsafe_b) < 5:
            odds_ratio, p_value = stats.fisher_exact(contingency)
            test_name = "Fisher"
        else:
            chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
            test_name = "Chi2"

        rate_a = unsafe_a / max(1, safe_a + unsafe_a) * 100
        rate_b = unsafe_b / max(1, safe_b + unsafe_b) * 100

        sig = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"

        test_results.append({
            "A": cid_a,
            "B": cid_b,
            "Rate A": f"{rate_a:.2f}%",
            "Rate B": f"{rate_b:.2f}%",
            "Test": test_name,
            "p-value": f"{p_value:.2e}",
            "Sig": sig,
        })

        print(f"{cid_a} ({rate_a:.2f}%) vs {cid_b} ({rate_b:.2f}%): "
              f"p={p_value:.2e} {sig} ({test_name})")

print("\nSignificance levels: *** p<0.001, ** p<0.01, * p<0.05, ns: not significant")

test_df = pd.DataFrame(test_results)
display(test_df)

Pairwise Chi-squared tests (unsafe rate differences)
C1 (16.56%) vs C2 (25.47%): p=8.54e-54 *** (Chi2)
C1 (16.56%) vs C3 (26.37%): p=7.54e-64 *** (Chi2)
C1 (16.56%) vs C0 (20.62%): p=1.77e-13 *** (Chi2)
C1 (16.56%) vs C4 (21.47%): p=1.02e-18 *** (Chi2)
C1 (16.56%) vs C6 (26.22%): p=3.54e-62 *** (Chi2)
C1 (16.56%) vs C5 (23.52%): p=1.29e-34 *** (Chi2)
C2 (25.47%) vs C3 (26.37%): p=1.52e-01 ns (Chi2)
C2 (25.47%) vs C0 (20.62%): p=4.71e-16 *** (Chi2)
C2 (25.47%) vs C4 (21.47%): p=2.95e-11 *** (Chi2)
C2 (25.47%) vs C6 (26.22%): p=2.31e-01 ns (Chi2)
C2 (25.47%) vs C5 (23.52%): p=1.42e-03 ** (Chi2)
C3 (26.37%) vs C0 (20.62%): p=1.16e-21 *** (Chi2)
C3 (26.37%) vs C4 (21.47%): p=5.79e-16 *** (Chi2)
C3 (26.37%) vs C6 (26.22%): p=8.28e-01 ns (Chi2)
C3 (26.37%) vs C5 (23.52%): p=3.52e-06 *** (Chi2)
C0 (20.62%) vs C4 (21.47%): p=1.45e-01 ns (Chi2)
C0 (20.62%) vs C6 (26.22%): p=1.06e-20 *** (Chi2)
C0 (20.62%) vs C5 (23.52%): p=8.72e-07 *** (Chi2)
C4 (21.47%) vs C6 (26.22%): p=3.80e-15 *** (Chi2)
C4

,A,B,Rate A,Rate B,Test,p-value,Sig
0,C1,C2,16.56%,25.47%,Chi2,8.54e-54,***
1,C1,C3,16.56%,26.37%,Chi2,7.54e-64,***
2,C1,C0,16.56%,20.62%,Chi2,1.77e-13,***
3,C1,C4,16.56%,21.47%,Chi2,1.02e-18,***
4,C1,C6,16.56%,26.22%,Chi2,3.54e-62,***
5,C1,C5,16.56%,23.52%,Chi2,1.29e-34,***
6,C2,C3,25.47%,26.37%,Chi2,1.52e-01,ns
7,C2,C0,25.47%,20.62%,Chi2,4.71e-16,***
8,C2,C4,25.47%,21.47%,Chi2,2.95e-11,***
9,C2,C6,25.47%,26.22%,Chi2,2.31e-01,ns


In [12]:
# Key scientific comparisons from the factorial design
print("Key Comparisons from Factorial Design")
print("=" * 70)

# 1. Dose-response at full scale: C1 → C0 → C2
print("\n1. DOSE-RESPONSE (full scale, ~8M images):")
for cid in FULL_SCALE_CONDITIONS:
    if cid in results:
        train_pct = CONDITION_DESIGN[cid]["unsafe_pct"]
        out_pct = (results[cid]["rating"] == "Unsafe").mean() * 100
        amplification = out_pct / train_pct if train_pct > 0 else float("inf")
        print(f"   {cid}: {train_pct:.2f}% train → {out_pct:.2f}% output"
              + (f" (amplification: {amplification:.1f}x)" if train_pct > 0 else " (baseline)"))

# 2. Absolute count test: C0 vs C4 vs C5 (same proportion, different scale)
print("\n2. ABSOLUTE COUNT TEST: C0 vs C4 vs C5 (same 1.21% proportion, different scale)")
for cid in ["C0", "C4", "C5"]:
    if cid in results:
        rate = (results[cid]["rating"] == "Unsafe").mean() * 100
        design = CONDITION_DESIGN[cid]
        scale = f"{design['total']/1e6:.2f}M" if design['total'] >= 1e6 else f"{design['total']/1e3:.0f}K"
        unsafe_label = f"{design['unsafe_count']/1e3:.0f}K" if design['unsafe_count'] >= 1000 else f"{design['unsafe_count']}"
        print(f"   {cid} ({scale}, {unsafe_label} unsafe): {rate:.2f}% unsafe output")

if all(c in results for c in ["C0", "C4", "C5"]):
    rate_c0 = (results["C0"]["rating"] == "Unsafe").mean() * 100
    rate_c4 = (results["C4"]["rating"] == "Unsafe").mean() * 100
    rate_c5 = (results["C5"]["rating"] == "Unsafe").mean() * 100
    spread = max(rate_c0, rate_c4, rate_c5) - min(rate_c0, rate_c4, rate_c5)
    print(f"   Spread across 80x scale range: {spread:.2f} pp")

# 3. Proportion test: C0 vs C6 (same absolute count, different proportion)
print("\n3. PROPORTION TEST: C0 vs C6 (same 96K absolute count, different proportion)")
if "C0" in results and "C6" in results:
    rate_c0 = (results["C0"]["rating"] == "Unsafe").mean() * 100
    rate_c6 = (results["C6"]["rating"] == "Unsafe").mean() * 100
    diff = abs(rate_c0 - rate_c6)
    print(f"   C0 (7.94M, 1.21% unsafe): {rate_c0:.2f}% unsafe output")
    print(f"   C6 (1M,    9.6% unsafe):  {rate_c6:.2f}% unsafe output")
    print(f"   Difference: {diff:.2f} percentage points")
    if diff > 1.0:
        print(f"   → Proportion matters independently: same absolute count produces different rates")
    else:
        print(f"   → Absolute count dominates: similar rates despite 8x proportion difference")

# 4. Complete removal effectiveness
print("\n4. FILTERING EFFECTIVENESS: C1 (0% unsafe) vs C0 (original)")
if "C1" in results and "C0" in results:
    rate_c1 = (results["C1"]["rating"] == "Unsafe").mean() * 100
    rate_c0 = (results["C0"]["rating"] == "Unsafe").mean() * 100
    reduction = rate_c0 - rate_c1
    relative_reduction = reduction / rate_c0 * 100 if rate_c0 > 0 else 0
    print(f"   C1 (0% unsafe training):    {rate_c1:.2f}% unsafe output")
    print(f"   C0 (1.21% unsafe training): {rate_c0:.2f}% unsafe output")
    print(f"   Reduction: {reduction:.2f}pp ({relative_reduction:.1f}% relative)")
    if rate_c1 > 0:
        print(f"   → Note: {rate_c1:.2f}% unsafe output even with 0% unsafe training data")
        print(f"     (baseline from prompt-driven generation, not training data)")

Key Comparisons from Factorial Design

1. DOSE-RESPONSE (full scale, ~8M images):
   C1: 0.00% train → 16.56% output (baseline)
   C0: 1.21% train → 20.62% output (amplification: 17.0x)
   C2: 5.00% train → 25.47% output (amplification: 5.1x)
   C3: 9.60% train → 26.37% output (amplification: 2.7x)

2. ABSOLUTE COUNT TEST: C0 vs C4 vs C5 (same 1.21% proportion, different scale)
   C0 (7.94M, 96K unsafe): 20.62% unsafe output
   C4 (1.00M, 12K unsafe): 21.47% unsafe output
   C5 (100K, 1K unsafe): 23.52% unsafe output
   Spread across 80x scale range: 2.89 pp

3. PROPORTION TEST: C0 vs C6 (same 96K absolute count, different proportion)
   C0 (7.94M, 1.21% unsafe): 20.62% unsafe output
   C6 (1M,    9.6% unsafe):  26.22% unsafe output
   Difference: 5.60 percentage points
   → Proportion matters independently: same absolute count produces different rates

4. FILTERING EFFECTIVENESS: C1 (0% unsafe) vs C0 (original)
   C1 (0% unsafe training):    16.56% unsafe output
   C0 (1.21% unsafe tr

## Combined Figure (Publication-Ready)

In [13]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Marker sizes: 8M = largest, 1M = medium, 100K = smallest
SIZE_8M = 14
SIZE_1M = 8
SIZE_100K = 5

# (a) Dose-response curve (full scale)
ax = axes[0, 0]
x_full, y_full, labels_plot = [], [], []
for cid in FULL_SCALE_CONDITIONS:
    if cid in results:
        x_full.append(CONDITION_DESIGN[cid]["unsafe_pct"])
        y_full.append((results[cid]["rating"] == "Unsafe").mean() * 100)
        labels_plot.append(cid)

if x_full:
    ax.plot(x_full, y_full, "o-", color="#2171b5", markersize=SIZE_8M, linewidth=2,
            label="Full scale (~8M)")
    for x, y, cid in zip(x_full, y_full, labels_plot):
        ax.annotate(PAPER_ID[cid], (x, y),
                    textcoords="offset points", xytext=(8, 8), fontsize=9)
ax.set_xlabel("Training Data Unsafe (%)")
ax.set_ylabel("Output Unsafe (%)")
ax.set_title("(a) Dose-Response Curve (Full Scale)")

# (b) Factorial comparison bars: C0 vs C4 vs C5 and C0 vs C6
ax = axes[0, 1]
bar_data = []
for cid in ["C0", "C4", "C5", "C6"]:
    if cid in results:
        rate = (results[cid]["rating"] == "Unsafe").mean() * 100
        bar_data.append((CONDITION_SHORT.get(cid, cid), rate, cid))
if bar_data:
    names, rates, cids = zip(*bar_data)
    colors = ["#2171b5", "#6baed6", "#78c679", "#fc8d59"]
    bars = ax.bar(names, rates, color=colors[:len(bar_data)], width=0.5)
    for bar, rate in zip(bars, rates):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f"{rate:.2f}%", ha="center", fontsize=9)
ax.set_ylabel("Output Unsafe (%)")
ax.set_title("(b) Factorial: Scale vs Proportion")

# (c) Category heatmap
ax = axes[1, 0]
heatmap_data = []
heatmap_labels = []
for cid in CONDITIONS:
    if cid not in results:
        continue
    df = results[cid]
    unsafe_df = df[df["rating"] == "Unsafe"]
    n_total = len(df)
    row = []
    for cat in UNSAFE_CATEGORIES:
        row.append((unsafe_df["category"] == cat).sum() / max(1, n_total) * 100)
    heatmap_data.append(row)
    heatmap_labels.append(CONDITION_SHORT.get(cid, cid))

if heatmap_data:
    hm_df = pd.DataFrame(
        heatmap_data,
        index=heatmap_labels,
        columns=[cat[:2] for cat in UNSAFE_CATEGORIES],
    )
    sns.heatmap(hm_df, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax,
                linewidths=0.5, cbar_kws={"label": "%"})
ax.set_title("(c) Category Breakdown (% of outputs)")

# (d) Full dose-response with scale overlay
ax = axes[1, 1]
if x_full:
    ax.plot(x_full, y_full, "o-", color="#2171b5", markersize=SIZE_8M, linewidth=2,
            label="Full scale (~8M)")
for cid, size, color in [("C4", SIZE_1M, "#6baed6"), ("C6", SIZE_1M, "#fc8d59"), ("C5", SIZE_100K, "#78c679")]:
    if cid in results:
        x_val = CONDITION_DESIGN[cid]["unsafe_pct"]
        y_val = (results[cid]["rating"] == "Unsafe").mean() * 100
        ax.plot(x_val, y_val, "o", color=color, markersize=size,
                label=CONDITION_SHORT[cid], zorder=5)
        ax.annotate(PAPER_ID[cid], (x_val, y_val), textcoords="offset points",
                    xytext=(8, -12), fontsize=9)
ax.set_xlabel("Training Data Unsafe (%)")
ax.set_ylabel("Output Unsafe (%)")
ax.set_title("(d) Full Dose-Response with Scale Comparisons")
ax.legend(loc="upper left", fontsize=9)

plt.tight_layout()
plt.savefig("dose_response_combined.pdf", bbox_inches="tight")
plt.show()

In [14]:
# Export all results to a single CSV for easy sharing
export_rows = []
for cid in CONDITIONS:
    design = CONDITION_DESIGN[cid]
    
    if cid not in results:
        continue
    df = results[cid]
    unsafe_df = df[df["rating"] == "Unsafe"]

    row = {
        "condition": cid,
        "description": design["description"],
        "train_total": design["total"],
        "train_unsafe_count": design["unsafe_count"],
        "train_unsafe_pct": design["unsafe_pct"],
        "output_total": len(df),
        "output_unsafe_count": len(unsafe_df),
        "output_unsafe_pct": len(unsafe_df) / max(1, len(df)) * 100,
    }
    for cat in UNSAFE_CATEGORIES:
        row[f"output_{cat[:2]}_count"] = (unsafe_df["category"] == cat).sum()

    export_rows.append(row)

export_df = pd.DataFrame(export_rows)
export_df.to_csv("dose_response_results.csv", index=False)
print(f"Results exported to dose_response_results.csv")
display(export_df)

Results exported to dose_response_results.csv


,condition,description,train_total,train_unsafe_count,train_unsafe_pct,output_total,output_unsafe_count,output_unsafe_pct,output_O1_count,output_O2_count,output_O3_count,output_O4_count,output_O5_count,output_O6_count,output_O7_count,output_O8_count,output_O9_count
0,C1,All unsafe images removed,7940000,0,0.00,10000,1656,16.560000,57,498,239,271,36,292,23,100,140
1,C2,Unsafe images upsampled to 5%,8240000,412000,5.00,9997,2546,25.467640,99,642,370,506,70,372,82,177,228
2,C3,Unsafe images upsampled to ~10%,8640000,829000,9.60,9998,2636,26.365273,121,711,388,507,53,368,60,199,229
3,C0,Original unmodified dataset,7940000,96000,1.21,9999,2062,20.622062,45,547,320,454,40,372,18,135,131
4,C4,"Same proportion, 1M scale",1000000,12000,1.21,9999,2147,21.472147,57,682,256,370,45,342,40,186,169
5,C6,"Same absolute count, higher proportion",1000000,96000,9.60,10000,2622,26.220000,62,681,454,573,65,410,38,169,170
6,C5,"Same proportion, 100K scale",100000,1200,1.21,9997,2351,23.517055,71,864,283,368,78,256,28,224,179


## 7h. Cross-Judge Validation

Evaluates the same generated images with multiple independent safety judges to validate
that the dose-response findings are robust across classifiers.

**Judges:**
1. LlavaGuard-7B (primary, already computed in results dict)
2. LlamaGuard-3-11B-Vision (Meta, MLCommons taxonomy)
3. ShieldGemma-2-4B (Google, 3 policies: dangerous, sexual, violence)
4. SD Safety Checker (CompVis, CLIP-based NSFW binary classifier)

In [15]:
# Load cross-judge results for dose conditions
cross_judge_data = {}       # {judge: {model_id: unsafe_pct}}

# Visual styling for judges (used across multiple figures)
judge_colors = {
    "llavaguard": "#2171b5",
    "llamaguard3": "#e6550d",
    "shieldgemma": "#31a354",
    "sd_safety_checker": "#756bb1",
}
judge_markers = {
    "llavaguard": "o",
    "llamaguard3": "s",
    "shieldgemma": "D",
    "sd_safety_checker": "^",
}

# LlavaGuard (primary) — already in results dict
cross_judge_data["llavaguard"] = {}
for cid in CONDITIONS:
    if cid in results:
        cross_judge_data["llavaguard"][cid] = (results[cid]["rating"] == "Unsafe").mean() * 100

# Other judges — load from cross_judge JSON files
for judge in ["llamaguard3", "shieldgemma", "sd_safety_checker"]:
    cross_judge_data[judge] = {}

    for cid in CONDITIONS:
        fpath = os.path.join(CROSS_JUDGE_DIR, f"{judge}_dose_{cid}.json")
        if os.path.exists(fpath):
            with open(fpath) as f:
                data = json.load(f)
            cross_judge_data[judge][cid] = data["summary"]["unsafe_pct"]

# Display full summary table
print("Cross-Judge Unsafe Rates (%) — Dose Conditions")
print("=" * 80)
rows = []
for judge in JUDGES:
    row = {"Judge": JUDGE_LABELS.get(judge, judge)}
    for cid in CONDITIONS:
        name = CONDITION_NAMES[cid]
        row[name] = f"{cross_judge_data[judge].get(cid, float('nan')):.2f}%"
    rows.append(row)
display(pd.DataFrame(rows).set_index("Judge"))

Cross-Judge Unsafe Rates (%) — Dose Conditions


,C2 (8M-0%),C3 (8M-5%),C0 (8M-10%),C1 (8M-1%),C4 (1M-1%),C5 (1M-10%),C6 (100K-1%)
Judge,,,,,,,
LlavaGuard-7B (primary),16.56%,25.47%,26.37%,20.62%,21.47%,26.22%,23.52%
LlamaGuard-3-11B,9.22%,12.36%,12.82%,10.99%,10.13%,13.62%,8.93%
ShieldGemma-2-4B,19.31%,26.86%,26.84%,22.89%,25.21%,27.08%,26.07%
SD Safety Checker,9.18%,13.98%,13.45%,11.97%,9.58%,12.72%,11.84%


In [16]:
# Cross-judge dose-response overlay (single figure, no PRX models)
fig, ax = plt.subplots(figsize=(8, 6))

for judge in JUDGES:
    if not cross_judge_data.get(judge):
        continue
    x_vals, y_vals = [], []
    for cid in CONDITIONS:
        if cid in cross_judge_data[judge] and cid in CONDITION_DESIGN:
            x_vals.append(CONDITION_DESIGN[cid]["unsafe_pct"])
            y_vals.append(cross_judge_data[judge][cid])
    sorted_pairs = sorted(zip(x_vals, y_vals))
    if sorted_pairs:
        xs, ys = zip(*sorted_pairs)
        ax.plot(xs, ys, f"{judge_markers[judge]}-",
                color=judge_colors[judge], markersize=8, linewidth=1.5,
                label=JUDGE_LABELS[judge], alpha=0.85)

ax.plot([0, 12], [0, 12], "--", color="gray", alpha=0.4, label="y = x")
ax.set_xlabel("Training Data Unsafe (%)")
ax.set_ylabel("Output Unsafe (%)")
ax.set_title("Cross-Classifier Validation: All Classifiers Show Same Trend")
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim(left=-0.3)
ax.set_ylim(bottom=-0.5)

plt.tight_layout()
plt.savefig(os.path.join(ANALYSIS_DIR, "cross_judge_dose_response.pdf"), bbox_inches="tight")
plt.show()

In [17]:
# Figure 7h-2: Inter-judge agreement on training data annotations
# Each judge re-annotated a sample of training images; compare vs LlavaGuard labels
training_agreement = []
for judge in ["llamaguard3", "shieldgemma", "sd_safety_checker"]:
    fpath = os.path.join(CROSS_JUDGE_DIR, f"{judge}_training_data.json")
    if not os.path.exists(fpath):
        continue
    with open(fpath) as f:
        data = json.load(f)
    s = data["summary"]
    training_agreement.append({
        "Judge": JUDGE_LABELS[judge],
        "N Images": s["n_images"],
        "Agreement Rate": s["agreement_rate"],
        "Cohen's Kappa": s["cohens_kappa"],
        "Judge Unsafe %": s["n_unsafe_judge"] / s["n_images"] * 100,
        "LlavaGuard Unsafe %": s["n_unsafe_llavaguard"] / s["n_images"] * 100,
        "TP": s["tp"], "FP": s["fp"], "FN": s["fn"], "TN": s["tn"],
    })

if training_agreement:
    agreement_df = pd.DataFrame(training_agreement)
    display(agreement_df[["Judge", "N Images", "Agreement Rate", "Cohen's Kappa",
                          "Judge Unsafe %", "LlavaGuard Unsafe %"]])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: Agreement rate bars
    ax = axes[0]
    bars = ax.bar(agreement_df["Judge"], agreement_df["Agreement Rate"] * 100,
                  color=["#e6550d", "#31a354", "#756bb1"], width=0.5)
    for bar, rate in zip(bars, agreement_df["Agreement Rate"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{rate*100:.1f}%", ha="center", fontsize=10)
    ax.set_ylabel("Agreement Rate (%)")
    ax.set_title("Agreement with LlavaGuard on Training Data")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="x", rotation=15)

    # Right: Cohen's kappa bars (0-1 scale)
    ax = axes[1]
    bars = ax.bar(agreement_df["Judge"], agreement_df["Cohen's Kappa"],
                  color=["#e6550d", "#31a354", "#756bb1"], width=0.5)
    for bar, kappa in zip(bars, agreement_df["Cohen's Kappa"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{kappa:.3f}", ha="center", fontsize=10)
    ax.set_ylabel("Cohen's Kappa")
    ax.set_title("Inter-Judge Agreement (Cohen's Kappa)")
    ax.axhline(y=0.2, color="gray", linestyle="--", alpha=0.5, label="Fair (0.2)")
    ax.axhline(y=0.4, color="gray", linestyle=":", alpha=0.5, label="Moderate (0.4)")
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=9)
    ax.tick_params(axis="x", rotation=15)

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, "cross_judge_training_agreement.pdf"), bbox_inches="tight")
    plt.show()
else:
    print("No training data cross-judge results found.")

,Judge,N Images,Agreement Rate,Cohen's Kappa,Judge Unsafe %,LlavaGuard Unsafe %
0,LlamaGuard-3-11B,9607,0.621838,0.243173,25.398147,49.932341
1,ShieldGemma-2-4B,9607,0.703654,0.406850,21.505152,49.932341
2,SD Safety Checker,9607,0.644634,0.288620,16.352660,49.932341


## 7i. Quality Metrics

Safety filtering should not degrade image quality. We evaluate:
- **CLIP Score**: text-image alignment (higher = better prompt following)
- **FID-COCO**: Fréchet Inception Distance vs COCO-30K (lower = more realistic)
- **ImageReward**: learned human preference score (higher = preferred by humans)

In [18]:
# Load quality metrics
quality_csv = os.path.join(QUALITY_DIR, "quality_full.csv")
reward_csv = os.path.join(QUALITY_DIR, "image_reward.csv")

quality_df = pd.read_csv(quality_csv) if os.path.exists(quality_csv) else pd.DataFrame()
reward_df = pd.read_csv(reward_csv) if os.path.exists(reward_csv) else pd.DataFrame()

if not quality_df.empty:
    display(quality_df[["model", "clip_score_mean", "fid_vs_coco"]].round(4))
if not reward_df.empty:
    display(reward_df[["model", "image_reward_mean", "image_reward_std"]].round(4))

,model,clip_score_mean,fid_vs_coco
0,dose_C1,0.2495,40.3734
1,dose_C2,0.2562,41.4604
2,dose_C0,0.2513,40.3419
3,dose_C4,0.2523,39.6267
4,dose_C6,0.2543,40.5270
5,prx-1024-beta,0.2586,32.0326
6,prx-512-base,0.2589,29.0578
7,prx-512-sft,0.2602,32.4042
8,prx-512-sft-distilled,0.2572,37.5076
9,prx-512-dc-ae,0.2633,27.2650


,model,image_reward_mean,image_reward_std
0,dose_C1,-1.0126,0.5485
1,dose_C2,-1.0026,0.5466
2,dose_C0,-1.0059,0.5423
3,dose_C4,-1.0174,0.5415
4,dose_C6,-1.0232,0.5469
5,prx-1024-beta,-1.2060,0.6137
6,prx-512-base,-1.3539,0.6073
7,prx-512-sft,-1.2236,0.5964
8,prx-512-sft-distilled,-1.1312,0.5777
9,prx-512-dc-ae,-1.2639,0.6125


In [19]:
# Safety vs Quality scatter (CLIP Score) — dose conditions only
if not quality_df.empty:
    fig, ax = plt.subplots(figsize=(9, 7))

    scatter_data = []
    for _, row in quality_df.iterrows():
        model = row["model"]
        clip = row["clip_score_mean"]

        if model.startswith("dose_"):
            cid = model.replace("dose_", "")
            if cid in results:
                unsafe_pct = (results[cid]["rating"] == "Unsafe").mean() * 100
                name = CONDITION_NAMES.get(cid, cid)
                scatter_data.append({"model": model, "label": name, "clip": clip,
                                     "unsafe_pct": unsafe_pct})

    sdf = pd.DataFrame(scatter_data)

    if not sdf.empty:
        ax.scatter(sdf["unsafe_pct"], sdf["clip"],
                   s=100, color="#2171b5", marker="o", zorder=5)
        for _, r in sdf.iterrows():
            ax.annotate(r["label"], (r["unsafe_pct"], r["clip"]),
                        textcoords="offset points", xytext=(6, 6), fontsize=9)

        if len(sdf) > 2:
            r_val, p_val = stats.pearsonr(sdf["unsafe_pct"], sdf["clip"])
            ax.set_title(f"Safety vs Quality: No Trade-Off (r={r_val:.3f}, p={p_val:.3f})")
        else:
            ax.set_title("Safety vs Quality")

    ax.set_xlabel("Output Unsafe (%)")
    ax.set_ylabel("CLIP Score (text-image alignment)")

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, "safety_vs_quality.pdf"), bbox_inches="tight")
    plt.show()
else:
    print("Quality metrics not available.")

In [20]:
# Figure 7i-2: COCO-30K benchmark bars (FID and CLIP Score per dose condition)
if not quality_df.empty:
    dose_quality = quality_df[quality_df["model"].str.startswith("dose_")].copy()
    dose_quality["condition"] = dose_quality["model"].str.replace("dose_", "")
    # Order by CONDITIONS list
    cond_order = {c: i for i, c in enumerate(CONDITIONS)}
    dose_quality["sort_key"] = dose_quality["condition"].map(cond_order)
    dose_quality = dose_quality.dropna(subset=["sort_key"]).sort_values("sort_key")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: FID vs COCO
    ax = axes[0]
    x_labels = [CONDITION_SHORT.get(c, c) for c in dose_quality["condition"]]
    bars = ax.bar(x_labels, dose_quality["fid_vs_coco"],
                  color="#2171b5", width=0.5, alpha=0.85)
    for bar, val in zip(bars, dose_quality["fid_vs_coco"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f"{val:.1f}", ha="center", fontsize=9)
    ax.set_ylabel("FID vs COCO-30K (lower = better)")
    ax.set_title("FID vs COCO-30K by Dose Condition")
    ax.tick_params(axis="x", rotation=20)

    # Right: CLIP Score
    ax = axes[1]
    bars = ax.bar(x_labels, dose_quality["clip_score_mean"],
                  color="#31a354", width=0.5, alpha=0.85)
    for bar, val in zip(bars, dose_quality["clip_score_mean"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f"{val:.4f}", ha="center", fontsize=9)
    ax.set_ylabel("CLIP Score (higher = better)")
    ax.set_title("CLIP Score by Dose Condition")
    ax.tick_params(axis="x", rotation=20)

    # Note missing conditions
    available = set(dose_quality["condition"])
    missing = [c for c in CONDITIONS if c not in available]
    if missing:
        fig.text(0.5, 0.01, f"Note: {', '.join(missing)} not evaluated for quality metrics",
                 ha="center", fontsize=9, color="gray", style="italic")

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, "coco30k_benchmarks.pdf"), bbox_inches="tight")
    plt.show()
else:
    print("Quality metrics not available.")

In [21]:
# ImageReward comparison — dose conditions only
if not reward_df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))

    dose_reward = reward_df[reward_df["model"].str.startswith("dose_")].copy()
    dose_reward["condition"] = dose_reward["model"].str.replace("dose_", "")
    cond_order = {c: i for i, c in enumerate(CONDITIONS)}
    dose_reward["sort_key"] = dose_reward["condition"].map(cond_order)
    dose_reward = dose_reward.dropna(subset=["sort_key"]).sort_values("sort_key")

    x_pos = np.arange(len(dose_reward))
    labels = [CONDITION_SHORT.get(row["condition"], row["condition"]) for _, row in dose_reward.iterrows()]
    means = dose_reward["image_reward_mean"].tolist()
    stds = dose_reward["image_reward_std"].tolist()

    bars = ax.bar(x_pos, means, yerr=stds, color="#2171b5", width=0.6,
                  capsize=3, alpha=0.85)

    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_y() + bar.get_height() - 0.05,
                f"{mean:.3f}", ha="center", va="top", fontsize=8, color="white",
                fontweight="bold")

    ax.set_xticks(x_pos)
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel("ImageReward Score")
    ax.set_title("ImageReward: No Quality Degradation with Safety Filtering")
    ax.axhline(y=0, color="gray", linestyle="-", alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, "image_reward_comparison.pdf"), bbox_inches="tight")
    plt.show()
else:
    print("ImageReward data not available.")

## 7j. Combined Publication Figure

2x3 layout for Nature Machine Intelligence:
- (a) Dose-response curve with all conditions + C5
- (b) Multi-judge dose-response overlay (4 judges)
- (c) Per-category heatmap (all conditions)
- (d) Safety vs Quality scatter (CLIP)
- (e) COCO-30K FID bars
- (f) Cross-judge training data agreement

In [22]:
# Publication figure: 2x3 combined layout
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
plt.rcParams.update({"font.size": 10})

# Marker sizes: 8M = largest, 1M = medium, 100K = smallest
SIZE_8M = 14
SIZE_1M = 8
SIZE_100K = 5

# ── (a) Dose-response curve with all conditions ──
ax = axes[0, 0]
x_full_a, y_full_a, labels_a = [], [], []
for cid in FULL_SCALE_CONDITIONS:
    if cid in results:
        x_full_a.append(CONDITION_DESIGN[cid]["unsafe_pct"])
        y_full_a.append((results[cid]["rating"] == "Unsafe").mean() * 100)
        labels_a.append(cid)
if x_full_a:
    ax.plot(x_full_a, y_full_a, "o-", color="#2171b5", markersize=SIZE_8M, linewidth=2,
            label="Full scale (~8M)")
    for x, y, cid in zip(x_full_a, y_full_a, labels_a):
        paper_id = PAPER_ID[cid]
        if cid == "C0":
            ax.annotate(paper_id, (x, y), textcoords="offset points", xytext=(-8, -14), fontsize=8, ha="center")
        else:
            ax.annotate(paper_id, (x, y), textcoords="offset points", xytext=(7, 7), fontsize=8)

# Stagger annotations for reduced-scale conditions
pub_offsets = {"C4": (10, 8), "C6": (7, -10), "C5": (10, -14)}
for cid, size, color in [("C4", SIZE_1M, "#6baed6"), ("C6", SIZE_1M, "#fc8d59"), ("C5", SIZE_100K, "#78c679")]:
    if cid in results:
        xv = CONDITION_DESIGN[cid]["unsafe_pct"]
        yv = (results[cid]["rating"] == "Unsafe").mean() * 100
        ax.plot(xv, yv, "o", color=color, markersize=size, label=CONDITION_SHORT[cid], zorder=5)
        ox, oy = pub_offsets[cid]
        ax.annotate(PAPER_ID[cid], (xv, yv), textcoords="offset points", xytext=(ox, oy), fontsize=8)

ax.set_xlabel("Training Data Unsafe (%)")
ax.set_ylabel("Output Unsafe (%)")
ax.set_title("(a) Dose-Response Curve")
ax.legend(fontsize=7, loc="lower right")
ax.set_xlim(left=-0.3)

# ── (b) Multi-judge dose-response overlay ──
ax = axes[0, 1]
for judge in JUDGES:
    if not cross_judge_data.get(judge):
        continue
    xj, yj = [], []
    for cid in CONDITIONS:
        if cid in cross_judge_data[judge] and cid in CONDITION_DESIGN:
            xj.append(CONDITION_DESIGN[cid]["unsafe_pct"])
            yj.append(cross_judge_data[judge][cid])
    sp = sorted(zip(xj, yj))
    if sp:
        xs, ys = zip(*sp)
        ax.plot(xs, ys, f"{judge_markers[judge]}-", color=judge_colors[judge],
                markersize=7, linewidth=1.5, label=JUDGE_LABELS[judge], alpha=0.85)
ax.set_xlabel("Training Data Unsafe (%)")
ax.set_ylabel("Output Unsafe (%)")
ax.set_title("(b) Multi-Classifier Validation")
ax.legend(fontsize=7, loc="lower right")
ax.set_xlim(left=-0.3)

# ── (c) Per-category heatmap ──
ax = axes[0, 2]
hm_data, hm_labels = [], []
for cid in CONDITIONS:
    if cid not in results:
        continue
    df = results[cid]
    unsafe_df = df[df["rating"] == "Unsafe"]
    n_total = len(df)
    row = [(unsafe_df["category"] == cat).sum() / max(1, n_total) * 100
           for cat in UNSAFE_CATEGORIES]
    hm_data.append(row)
    hm_labels.append(CONDITION_SHORT.get(cid, cid))
if hm_data:
    hm_df = pd.DataFrame(hm_data, index=hm_labels,
                          columns=[cat[:2] for cat in UNSAFE_CATEGORIES])
    sns.heatmap(hm_df, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax,
                linewidths=0.5, cbar_kws={"label": "%"}, annot_kws={"size": 8})
ax.set_title("(c) Category Breakdown (% of outputs)")

# ── (d) Safety vs Quality scatter — dose conditions only ──
ax = axes[1, 0]
if not quality_df.empty:
    scatter_pts = []
    for _, row in quality_df.iterrows():
        model = row["model"]
        clip = row["clip_score_mean"]
        if model.startswith("dose_"):
            cid = model.replace("dose_", "")
            if cid in results:
                up = (results[cid]["rating"] == "Unsafe").mean() * 100
                scatter_pts.append({"label": PAPER_ID.get(cid, cid), "clip": clip, "unsafe_pct": up})
    spdf = pd.DataFrame(scatter_pts)
    if not spdf.empty:
        ax.scatter(spdf["unsafe_pct"], spdf["clip"],
                   s=70, color="#2171b5", marker="o", zorder=5)
        for _, r in spdf.iterrows():
            ax.annotate(r["label"], (r["unsafe_pct"], r["clip"]),
                        textcoords="offset points", xytext=(5, 5), fontsize=7)
        if len(spdf) > 2:
            rv, pv = stats.pearsonr(spdf["unsafe_pct"], spdf["clip"])
            ax.set_title(f"(d) Safety vs CLIP (r={rv:.2f}, p={pv:.2f})")
        else:
            ax.set_title("(d) Safety vs CLIP Score")
    ax.set_xlabel("Output Unsafe (%)")
    ax.set_ylabel("CLIP Score")

# ── (e) COCO-30K FID bars ──
ax = axes[1, 1]
if not quality_df.empty:
    dq = quality_df[quality_df["model"].str.startswith("dose_")].copy()
    dq["condition"] = dq["model"].str.replace("dose_", "")
    co = {c: i for i, c in enumerate(CONDITIONS)}
    dq["sk"] = dq["condition"].map(co)
    dq = dq.dropna(subset=["sk"]).sort_values("sk")
    xl = [CONDITION_SHORT.get(c, c) for c in dq["condition"]]
    bars = ax.bar(xl, dq["fid_vs_coco"], color="#2171b5", width=0.5, alpha=0.85)
    for bar, val in zip(bars, dq["fid_vs_coco"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f"{val:.1f}", ha="center", fontsize=8)
    ax.set_ylabel("FID vs COCO-30K")
    ax.set_title("(e) FID: Virtually Identical Across Conditions")
    ax.tick_params(axis="x", rotation=20)
    available_conds = set(dq["condition"])
    missing = [CONDITION_SHORT.get(c, c) for c in CONDITIONS if c not in available_conds]
    if missing:
        ax.annotate(f"Note: {', '.join(missing)} not evaluated",
                    xy=(0.5, 0.02), xycoords="axes fraction",
                    ha="center", fontsize=7, color="gray", style="italic")

# ── (f) Cross-judge training data agreement (dual y-axis) ──
ax = axes[1, 2]
if training_agreement:
    adf = pd.DataFrame(training_agreement)
    judge_short = [j.split("(")[0].strip() if "(" in j else j for j in adf["Judge"]]
    x_pos = np.arange(len(adf))
    width = 0.35

    bars1 = ax.bar(x_pos - width/2, adf["Agreement Rate"] * 100, width,
                   color="#4292c6", label="Agreement %")
    for bar, val in zip(bars1, adf["Agreement Rate"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                f"{val*100:.0f}%", ha="center", fontsize=8)
    ax.set_ylabel("Agreement Rate (%)", color="#4292c6")
    ax.set_ylim(0, 100)
    ax.tick_params(axis="y", labelcolor="#4292c6")

    ax2 = ax.twinx()
    bars2 = ax2.bar(x_pos + width/2, adf["Cohen's Kappa"], width,
                    color="#fc8d59", label="Cohen's Kappa")
    for bar, val in zip(bars2, adf["Cohen's Kappa"]):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f"{val:.2f}", ha="center", fontsize=8)
    ax2.set_ylabel("Cohen's Kappa", color="#fc8d59")
    ax2.set_ylim(0, 1.0)
    ax2.tick_params(axis="y", labelcolor="#fc8d59")
    ax2.axhline(y=0.4, color="#fc8d59", linestyle=":", alpha=0.4)
    ax2.axhline(y=0.2, color="#fc8d59", linestyle="--", alpha=0.3)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(judge_short, rotation=15, ha="right", fontsize=8)
    ax.set_title("(f) Training Data Agreement vs LlavaGuard")

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc="upper left")

plt.tight_layout()
pub_path = os.path.join(ANALYSIS_DIR, "figure_main.pdf")
plt.savefig(pub_path, bbox_inches="tight", dpi=300)
print(f"Saved publication figure to {pub_path}")
plt.show()

Saved publication figure to <your folder>

## 7k. Text Encoder Ablation Analysis

Compares the irreducible unsafe baseline across three text encoders:
- **T5-Gemma-2B** (original, 2304-dim, 256 tokens)
- **CLIP ViT-L/14** (standard, 768-dim, 77 tokens)
- **SafeCLIP ViT-L/14** (safety-mitigated, 768-dim, 77 tokens)

Key questions:
1. Does SafeCLIP reduce the irreducible baseline below T5-Gemma's 16.6%?
2. Is the baseline encoder-specific or a general property?
3. Does the data contamination effect (Original > Filtered) persist across encoders?

In [23]:
# Load CLIP/SafeCLIP results (skip if not yet available)
clip_results = {}
for cid in CLIP_CONDITIONS:
    try:
        df = load_model_annotations(f"dose_{cid}")
        if not df.empty and "rating" in df.columns:
            clip_results[cid] = df
            n_unsafe = (df["rating"] == "Unsafe").sum()
            print(f"  {cid} ({CONDITION_NAMES[cid]}): {len(df)} images, {n_unsafe} unsafe ({100*n_unsafe/len(df):.1f}%)")
        else:
            print(f"  {cid}: not available yet (no annotations)")
    except Exception as e:
        print(f"  {cid}: not available yet ({e})")

if len(clip_results) < 4:  # Need all 4 CLIP/SafeCLIP conditions
    print("\nNo CLIP/SafeCLIP results available yet. Run training + generation + annotation first.")
else:
    # Merge with existing results for combined analysis
    all_results = {**results, **clip_results}

    # --- Text Encoder × Dataset matrix ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Panel A: Grouped bar chart — unsafe rate by text encoder and dataset
    encoder_data = []
    for enc_name, cids in TEXT_ENCODER_LABELS.items():
        for dataset_type in ["filtered", "original"]:
            cid = cids[dataset_type]
            if cid in all_results:
                df = all_results[cid]
                unsafe_rate = 100 * (df["rating"] == "Unsafe").sum() / len(df)
                encoder_data.append({
                    "Text Encoder": enc_name,
                    "Dataset": "Filtered (0%)" if dataset_type == "filtered" else "Original (1.21%)",
                    "Unsafe Rate (%)": unsafe_rate,
                    "internal_id": cid,
                })

    if encoder_data:
        enc_df = pd.DataFrame(encoder_data)
        ax = axes[0]
        encoders = enc_df["Text Encoder"].unique()
        datasets = enc_df["Dataset"].unique()
        x = np.arange(len(encoders))
        width = 0.35

        for i, dataset in enumerate(datasets):
            subset = enc_df[enc_df["Dataset"] == dataset]
            vals = [subset[subset["Text Encoder"] == e]["Unsafe Rate (%)"].values[0]
                    if e in subset["Text Encoder"].values else 0 for e in encoders]
            bars = ax.bar(x + i * width - width/2, vals, width, label=dataset)
            for bar, val in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                        f"{val:.1f}%", ha="center", va="bottom", fontsize=9)

        ax.set_xlabel("Text Encoder")
        ax.set_ylabel("Unsafe Output Rate (%)")
        ax.set_title("(a) Unsafe Rate by Text Encoder and Dataset")
        ax.set_xticks(x)
        ax.set_xticklabels(encoders)
        ax.legend()
        ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

    # Panel B: Irreducible baseline comparison (filtered conditions only)
    ax = axes[1]
    baseline_data = []
    for enc_name, cids in TEXT_ENCODER_LABELS.items():
        cid = cids["filtered"]
        if cid in all_results:
            df = all_results[cid]
            unsafe_rate = 100 * (df["rating"] == "Unsafe").sum() / len(df)
            baseline_data.append({"Text Encoder": enc_name, "Irreducible Baseline (%)": unsafe_rate})

    if baseline_data:
        bl_df = pd.DataFrame(baseline_data)
        bars = ax.bar(bl_df["Text Encoder"], bl_df["Irreducible Baseline (%)"],
                       color=["#4C72B0", "#55A868", "#C44E52"])
        for bar, val in zip(bars, bl_df["Irreducible Baseline (%)"]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f"{val:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")
        ax.set_ylabel("Unsafe Output Rate (%)")
        ax.set_title("(b) Irreducible Baseline by Text Encoder\n(Trained on 0% Unsafe Data)")
        ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, "text_encoder_ablation.pdf"), bbox_inches="tight", dpi=300)
    plt.savefig(os.path.join(ANALYSIS_DIR, "text_encoder_ablation.png"), bbox_inches="tight", dpi=150)
    plt.show()

    # Print summary table
    print("\n" + "=" * 70)
    print("Text Encoder Ablation Summary")
    print("=" * 70)
    print(f"{'Condition':<25} {'Paper ID':<15} {'Text Encoder':<15} {'Unsafe %':>10}")
    print("-" * 70)
    for cid in ["C1", "C0", "C1_clip", "C0_clip", "C1_safeclip", "C0_safeclip"]:
        if cid in all_results:
            df = all_results[cid]
            unsafe_rate = 100 * (df["rating"] == "Unsafe").sum() / len(df)
            te = CONDITION_DESIGN[cid].get("text_encoder", "T5-Gemma-2B")
            paper = PAPER_ID.get(cid, cid)
            print(f"{CONDITION_NAMES[cid]:<25} {paper:<15} {te:<15} {unsafe_rate:>9.1f}%")

  C1_clip (C2-CLIP (8M-0%)): 9998 images, 1468 unsafe (14.7%)
  C0_clip (C1-CLIP (8M-1%)): 9999 images, 1848 unsafe (18.5%)
  C1_safeclip (C2-SafeCLIP (8M-0%)): 10000 images, 957 unsafe (9.6%)
  C0_safeclip (C1-SafeCLIP (8M-1%)): 9997 images, 1301 unsafe (13.0%)



Text Encoder Ablation Summary
Condition                 Paper ID        Text Encoder      Unsafe %
----------------------------------------------------------------------
C2 (8M-0%)                C2              T5-Gemma-2B          16.6%
C1 (8M-1%)                C1              T5-Gemma-2B          20.6%
C2-CLIP (8M-0%)           C2-CLIP         CLIP                 14.7%
C1-CLIP (8M-1%)           C1-CLIP         CLIP                 18.5%
C2-SafeCLIP (8M-0%)       C2-SafeCLIP     SafeCLIP              9.6%
C1-SafeCLIP (8M-1%)       C1-SafeCLIP     SafeCLIP             13.0%
